In [0]:
/*
================================================================================
VISTA: supermercadoetl_prod.gold.recetas
CAPA: Gold (Analítica - Cálculo de Recetas)
ENTORNO: PRODUCCIÓN
AUTOR: Sistema ETL Supermercados
FECHA CREACIÓN: 2026-04-17
ÚLTIMA MODIFICACIÓN: 2026-04-17
================================================================================

PROPÓSITO:
Vista que define recetas mediante ingredientes, cantidades necesarias y 
rendimiento. Se usa con matching inteligente contra gold.catalogo_productos 
para calcular costos totales y por porción automáticamente.

DEPENDENCIAS:
- Vista relacionada: gold.catalogo_productos (matching de ingredientes)
- Tabla consumidora: Sistema de cálculo de costos de recetas

ARQUITECTURA DEL SISTEMA:
┌─────────────────────────────────────────────────────────────────────────┐
│  silver.precios_productos                                               │
│    ↓                                                                    │
│  gold.catalogo_productos (normaliza cantidades → precio_unitario)      │
│    ↓                                                                    │
│  gold.recetas (define ingredientes y cantidades) ← ESTA VISTA          │
│    ↓                                                                    │
│  JOIN con matching inteligente                                         │
│    ↓                                                                    │
│  Cálculo: costo_ingrediente = precio_unitario × cantidad_necesaria    │
│    ↓                                                                    │
│  Agregación: costo_total y costo_por_porcion                          │
└─────────────────────────────────────────────────────────────────────────┘

ESTRUCTURA DE DATOS:
┌──────────────────────────┬──────────────┬─────────────────────────────────┐
│ Campo                    │ Tipo         │ Descripción                     │
├──────────────────────────┼──────────────┼─────────────────────────────────┤
│ receta                   │ STRING       │ Nombre de la receta             │
│ ingrediente              │ STRING       │ Nombre SIMPLIFICADO del producto│
│ cantidad_necesaria       │ DECIMAL      │ Cantidad del ingrediente        │
│ unidad_receta            │ STRING       │ Unidad (gramos/ml/unidades)     │
│ rendimiento_porciones    │ INT          │ Número de porciones que rinde   │
└──────────────────────────┴──────────────┴─────────────────────────────────┘

REGLAS DE NEGOCIO:

1. MATCHING DE INGREDIENTES (con gold.catalogo_productos):
   - Campo 'ingrediente' usa nombres SIMPLIFICADOS para matching flexible
   - El sistema busca similitud usando LIKE y scoring
   - Ejemplos de matching exitoso:
     * 'Harina Trigo 0000' → matchea 'Harina Trigo 0000 MORIXE 1kg'
     * 'Queso Mozzarella' → matchea 'Queso Mozzarella BARRAZA 1kg'
     * 'Maple Huevo' → matchea 'Huevo Blanco Maple X 30u'
     * 'Ajo' → matchea 'Ajo' (exacto)

2. UNIDADES DE MEDIDA (debe coincidir con catálogo):
   - 'gramos': Ingredientes sólidos (harina, sal, queso, carne)
   - 'ml' o 'mililitros': Ingredientes líquidos (leche, aceite, crema)
   - 'unidades': Productos discretos (huevos, ajos, tapas, latas)

3. CANTIDADES FRACCIONARIAS:
   - Se permiten decimales para sub-unidades
   - Ejemplo: 0.167 unidades = 1/6 de ajo
   - Ejemplo: 5 unidades de 'Maple Huevo' = 5 huevos individuales
     * El catálogo calcula: cantidad_base = 30 (maple completo)
     * El costo se calcula: 5 × (precio_maple / 30) = precio de 5 huevos

4. RENDIMIENTO DE RECETAS:
   - Define cuántas porciones produce la receta completa
   - Usado para calcular: costo_por_porcion = costo_total / rendimiento
   - Ejemplos:
     * Pizza: rendimiento 2 = costo para 2 pizzas individuales
     * Empanadas: rendimiento 3 = costo para 3 docenas (36 unidades)
     * Porción = unidad de medida relevante para cada receta

5. ORGANIZACIÓN DE DATOS:
   - Ingredientes agrupados por receta
   - Orden sugerido: ingredientes base → condimentos → toppings
   - Comentarios inline documentan rendimiento (ej: "-- rinde 2 pizzas")

CÁLCULO AUTOMÁTICO DEL SISTEMA:
Cuando se joinea con gold.catalogo_productos:
  
  1. Matching: Busca ingrediente en catálogo (por nombre similar)
  2. Obtiene: precio_unitario del catálogo ($/gramo, $/ml, $/unidad)
  3. Multiplica: costo_ingrediente = precio_unitario × cantidad_necesaria
  4. Suma: costo_total = SUM(costo_ingrediente) por receta
  5. Divide: costo_por_porcion = costo_total / rendimiento_porciones

RECETAS DEFINIDAS:
┌─────────────┬──────────────────┬─────────────────────────────────────────┐
│ Receta      │ Rendimiento      │ Descripción                             │
├─────────────┼──────────────────┼─────────────────────────────────────────┤
│ Pizza       │ 2 pizzas         │ 10 ingredientes, base de harina con     │
│             │                  │ salsa de tomate y queso                 │
├─────────────┼──────────────────┼─────────────────────────────────────────┤
│ Empanadas   │ 3 docenas        │ 11 ingredientes, relleno de carne con   │
│             │ (36 unidades)    │ verduras salteadas                      │
└─────────────┴──────────────────┴─────────────────────────────────────────┘

CASOS DE USO:
- Cálculo de costo de producción para venta
- Planificación de compras para eventos (ej: fiesta de 50 personas)
- Análisis de margen de ganancia por producto elaborado
- Comparación de costos entre recetas alternativas
- Tracking de variación de costos en el tiempo (histórico de precios)

MÉTRICAS ESPERADAS:
- Recetas activas: 2 (Pizza, Empanadas)
- Ingredientes únicos: ~20-25 productos
- Costo promedio por porción: $5,000 - $10,000 ARS
- Actualización: On-demand (agregar nuevas recetas manualmente)

VALIDACIONES DE CALIDAD:
✓ Todos los ingredientes deben existir en gold.catalogo_productos
✓ cantidad_necesaria > 0 para todos los ingredientes
✓ rendimiento_porciones > 0 para todas las recetas
✓ unidad_receta debe ser válida (gramos/ml/unidades)
✓ Matching debe tener score > 0.5 (similitud mínima)

AGREGAR NUEVAS RECETAS:
Para agregar una receta, simplemente agregar filas al VALUES:

```sql
-- Nueva receta: Salsa Bechamel (rinde 4 porciones)
('Salsa Bechamel', 'Leche Entera', 500, 'ml', 4),
('Salsa Bechamel', 'Harina Trigo 0000', 50, 'gramos', 4),
('Salsa Bechamel', 'Manteca', 50, 'gramos', 4),
('Salsa Bechamel', 'Sal Fina', 5, 'gramos', 4)
```

MANTENIMIENTO:
- Para modificar cantidades: editar cantidad_necesaria
- Para cambiar rendimiento: editar rendimiento_porciones
- Para agregar ingrediente: agregar nueva fila en VALUES
- Para quitar ingrediente: remover fila del VALUES
- Verificar que nombres matcheen con gold.catalogo_productos

CONSULTA DE EJEMPLO:
```sql
-- Calcular costo de recetas con precios actuales
WITH calculo_costos AS (
  SELECT 
    r.receta,
    r.ingrediente,
    c.precio_unitario,
    r.cantidad_necesaria,
    c.precio_unitario * r.cantidad_necesaria AS costo_ingrediente,
    r.rendimiento_porciones
  FROM gold.recetas r
  INNER JOIN gold.catalogo_productos c
    ON LOWER(c.nombre) LIKE CONCAT('%', LOWER(r.ingrediente), '%')
  WHERE c.fecha_extraccion = CURRENT_DATE()
)
SELECT 
  receta,
  ROUND(SUM(costo_ingrediente), 2) AS costo_total,
  MAX(rendimiento_porciones) AS porciones,
  ROUND(SUM(costo_ingrediente) / MAX(rendimiento_porciones), 2) AS costo_por_porcion
FROM calculo_costos
GROUP BY receta;
```

VENTAJAS DEL DISEÑO:
✅ Desacoplado del catálogo (independiente de cambios en precios)
✅ Escalable (agregar recetas sin modificar lógica)
✅ Mantenible (estructura simple VALUES)
✅ Flexible (soporta diferentes tipos de medida)
✅ Auditable (histórico de costos usando fecha_extraccion)

SIGUIENTE PASO EN ARQUITECTURA:
Esta vista se joinea con gold.catalogo_productos en queries de análisis
para generar reportes de costos actualizados automáticamente.

NOTAS TÉCNICAS:
- VALUES permite definir datos estáticos directamente en SQL
- Alias 't' nombra las columnas del VALUES para claridad
- Compatible con cualquier herramienta SQL estándar
- No requiere tabla física (es una vista virtual)

CHANGELOG:
- 2026-04-17: Creación inicial con 2 recetas (Pizza, Empanadas)
- 2026-04-17: Documentadas 21 líneas de ingredientes
- 2026-04-17: Agregado soporte para cantidades fraccionarias (ajo 1/6)

================================================================================
*/

-- ============================================
-- CREACIÓN DE VISTA: recetas
-- ============================================

CREATE OR REPLACE VIEW supermercadoetl_prod.gold.recetas AS
  SELECT * FROM VALUES
    -- ─────────────────────────────────────────────────────────────
    -- RECETA: Pizza (rinde 2 pizzas)
    -- Costo estimado: ~$13,000 ARS → $6,500/pizza
    -- ───────────────────────────────────────────────────────────── 
    -- 2 Pizzas

    -- - Harina 500 gr
    -- - condimento pizza 3 gr
    -- - sal 10 gr
    -- - pimenton 2 gr
    -- - Oregano 5 gr
    -- - 1 lata tomate
    -- - 1 sobre levadura
    -- - Aji molido 2 gr
    -- - 1/6 ajo
    -- - Queso mozzarela 600 gr

    -- ─────────────────────────────────────────────────────────────
    ('Pizza', 'Harina Trigo 0000', 500, 'gramos', 2),
    ('Pizza', 'Condimento Para Pizza', 3, 'gramos', 2),
    ('Pizza', 'Sal Fina', 10, 'gramos', 2),
    ('Pizza', 'Pimenton', 2, 'gramos', 2),
    ('Pizza', 'Orégano', 5, 'gramos', 2),
    ('Pizza', 'Tomate Perita en lata', 1, 'unidades', 2),
    ('Pizza', 'Levadura Seca', 1, 'unidades', 2),
    ('Pizza', 'Aji Molido', 2, 'gramos', 2),
    ('Pizza', 'Ajo', 0.167, 'unidades', 2), -- 1/6 de ajo
    ('Pizza', 'Queso Mozzarella', 600, 'gramos', 2),
    
    -- ─────────────────────────────────────────────────────────────
    -- RECETA: Empanadas (rinde 3 docenas = 36 empanadas)
    -- Costo estimado: ~$29,000 ARS → $9,688/docena → $807/empanada
    -- ─────────────────────────────────────────────────────────────
    
    -- 3 DOCENAS EMPANADAS

    -- - 1kg carne picada
    -- - 3 paquetes de tapas de empanadas x12
    -- - 2 atados puerro
    -- - 2 atados cebolla de verdeo
    -- - 1 cebolla
    -- - 2 zanahorias
    -- - 1 morron
    -- - sal
    -- - condimento empanadas
    -- - 1 paquete de aceitunas (300 gr)
    -- - 5 huevos

    -- ─────────────────────────────────────────────────────────────

    ('Empanadas', 'Picada Desgrasada', 1000, 'gramos', 3),
    ('Empanadas', 'Tapa Para Empanadas X12', 3, 'unidades', 3),
    ('Empanadas', 'Cebolla', 200, 'gramos', 3),
    ('Empanadas', 'Puerro', 2, 'unidades', 3),
    ('Empanadas', 'Cebolla de verdeo', 2, 'unidades', 3),
    ('Empanadas', 'Zanahoria', 150, 'gramos', 3),
    ('Empanadas', 'Morron rojo', 250, 'gramos', 3),
    ('Empanadas', 'Sal Fina', 50, 'gramos', 3),
    ('Empanadas', 'Condimento Para Empanadas', 5, 'gramos', 3),
    ('Empanadas', 'Aceitunas Verdes Descarozadas', 300, 'unidades', 3),
    ('Empanadas', 'Maple Huevo', 5, 'unidades', 3), -- 5 huevos individuales del maple

    -- ─────────────────────────────────────────────────────────────

    -- 1kg milanesas 
    -- - 1kg Bola de Lomo   
    -- - 1 Perejil 
    -- - 1 Ajo 
    -- - 500 gr Pan Rallado
    -- - 100 gr Harina 0000
    -- - Sal fina 50 gramos
    -- - 5 huevos   

    ('Milanesas', 'Bola de Lomo', 1000, 'gramos', 1),
    ('Milanesas', 'Sal Fina', 50, 'gramos', 1),
    ('Milanesas', 'Maple Huevo', 5, 'unidades', 1),
    ('Milanesas', 'Pan Rallado', 500, 'gramos', 1),
    ('Milanesas', 'Perejil', 1, 'unidades', 1),
    ('Milanesas', 'Ajo', 1, 'unidades', 1),
    ('Milanesas', 'Harina Trigo 0000', 100, 'gramos', 1)
    
  AS t(receta, ingrediente, cantidad_necesaria, unidad_receta, rendimiento_porciones)